In [1]:
import torch
import torch.nn as nn
import pandas as pd
import torchvision
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from torch.optim import Adam
from torchvision.transforms import transforms
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader,Dataset
from torchvision import models

In [2]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [10]:
import os

train_df=pd.read_csv(r"train\train.csv")
val_df=pd.read_csv(r"val\val.csv")
# val_df=pd.read_csv("val.csv")

In [12]:
train_df["category"].unique()

array([0, 1, 2])

In [46]:
import torchvision.transforms as transforms
from torchvision import transforms

In [47]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.to(torch.float))
])

In [60]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transforms=None):
        self.dataframe = dataframe.reset_index(drop=True)  # ✅ reset index to avoid alignment issues
        self.transforms = transforms
        self.labels = torch.tensor(self.dataframe["category"].values, dtype=torch.long)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]  # ✅ Use iloc for row position
        image, label = row["image"], row["label"]  # Adjust column names as needed

        if self.transforms:
            image = self.transforms(image)
            return image, label


In [17]:
# my_transforms = transforms.Compose([
#     transforms.Resize((28,28)),
#     transforms.ToTensor(),  # already scales to [0,1], no need /255
# ])
# dataset = CustomImageDataset(train_df, transform=my_transforms, device='cpu')

In [61]:
train_dataset=CustomImageDataset(dataframe=train_df,transforms=transforms)
val_dataset=CustomImageDataset(dataframe=val_df,transforms=transforms)

In [62]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.to(torch.float))  # optional float cast
])

In [64]:
# n_rows, n_cols = 3, 3
# fig, axarr = plt.subplots(n_rows, n_cols, figsize=(6, 6))

# for row in range(n_rows):
#     for col in range(n_cols):
#         idx = np.random.randint(0, len(train_dataset))
#         image = train_dataset[idx][0].cpu()

#         # Scale image to [0, 255]
#         image = image * 255.0

#         # Handle grayscale and RGB
#         if image.shape[0] == 1:
#             image = image.squeeze(0).numpy()  # [H, W]
#             axarr[row, col].imshow(image.astype(np.uint8), cmap='gray')
#         else:
#             image = image.permute(1, 2, 0).numpy()  # [H, W, C]
#             axarr[row, col].imshow(image.astype(np.uint8))

#         axarr[row, col].axis('off')

# plt.tight_layout()
# plt.show()

In [33]:
LR=1e-3
BATCH_SIZE=4
EPOCHS=15
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
## Transfer Learning

In [34]:
googlenet_model=models.googlenet(weights='DEFAULT')
for param in googlenet_model.parameters():
    param.requires_grad=True

In [35]:
num_classes=len(train_df["category"].unique())
googlenet_model.fc=torch.nn.Linear(googlenet_model.fc.in_features,num_classes)
googlenet_model.fc

Linear(in_features=1024, out_features=3, bias=True)

In [36]:
googlenet_model.to(device)

GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track

In [37]:

my_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # GoogLeNet expects 224x224
    transforms.ToTensor(),
])
# dataset = CustomImageDataset(df, transform=my_transforms)
val_loader = torch.utils.data.DataLoader(val_df, batch_size=32, shuffle=False)

In [38]:
val_dataset = CustomImageDataset(val_df, transforms=my_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [39]:
googlenet_model = googlenet_model.to(device)
googlenet_model.eval()  # evaluation mode

total_acc_test = 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        predictions = googlenet_model(inputs)
        acc = (predictions.argmax(dim=1) == labels).sum().item()
        total_acc_test += acc

val_accuracy = total_acc_test / len(val_dataset)
print("Validation accuracy:", val_accuracy)


Validation accuracy: 0.40601503759398494


In [40]:
criterion = torch.nn.CrossEntropyLoss()
total_loss_test = 0
total_acc_test = 0

googlenet_model.eval()
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predictions = googlenet_model(inputs)
        loss = criterion(predictions, labels)

        total_loss_test += loss.item() * inputs.size(0)
        total_acc_test += (torch.argmax(predictions, dim=1) == labels).sum().item()

val_loss = total_loss_test / len(val_dataset)
val_accuracy = total_acc_test / len(val_dataset)
print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}")

Validation Loss: 1.1093, Accuracy: 0.4060


In [41]:
print(val_df.columns.tolist())
print(val_df.head())

['image:FILE', 'category']
                       image:FILE  category
0  val/healthy/healthy_val.25.jpg         0
1  val/healthy/healthy_val.32.jpg         0
2   val/healthy/healthy_val.3.jpg         0
3  val/healthy/healthy_val.16.jpg         0
4  val/healthy/healthy_val.10.jpg         0


In [42]:
criterion = torch.nn.CrossEntropyLoss()
total_loss_test = 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predictions = googlenet_model(inputs)
        loss = criterion(predictions, labels)
        total_loss_test += loss.item() * inputs.size(0)
        total_acc_test += (torch.argmax(predictions, dim=1) == labels).sum().item()

val_loss = total_loss_test / len(val_dataset)
val_accuracy = total_acc_test / len(val_dataset)

print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.4f}")

Validation Loss: 1.1093, Accuracy: 0.8120


In [ ]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data, transforms=None):
        self.data = data
        self.transforms = transforms

    def __getitem__(self, idx):
        image, label = self.data[idx]
        if self.transforms:
            image = self.transforms(image)  
        return image, label

    def __len__(self):
        return len(self.data)

In [ ]:
# from torchvision import transforms

# transform_pipeline = transforms.Compose([
#     transforms.Resize((128, 128)),
#     transforms.ToTensor()
# ])

# train_dataset = MyDataset(train_df, transforms=transform_pipeline)  module

In [1]:
# sample = train_dataset[idx]
# print(type(sample), len(sample))